# Single Agent, Multi-Tool Call

A LangGraph agent that **plans, calls tools, recovers from failures and writes a clean report**.

Demo task: *"Is a Burj Khalifa (At The Top) visit worth it for 2 adults?"*
The agent checks live Dubai weather, searches ticket prices, fetches the AED to PKR rate,
calculates the budget and returns a recommendation.

**Before you run:** install the requirements and set `GOOGLE_API_KEY`, `OPENWEATHER_API_KEY`
and `TAVILY_API_KEY` (Kaggle Secrets or a local `.env` file). See the README for details.

In [ ]:
# ===========================yha sa ha yeh scene========================
%pip install -U langgraph langchain-core langchain-google-genai pydantic requests tavily-python python-dotenv

In [ ]:
import os

KEYS = ["GOOGLE_API_KEY", "OPENWEATHER_API_KEY", "TAVILY_API_KEY"]

try:
    # Kaggle: keys come from Add-ons > Secrets
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    for name in KEYS:
        os.environ[name] = user_secrets.get_secret(name)

except ImportError:
    # Local machine / GitHub Codespaces: keys come from the .env file
    from dotenv import load_dotenv

    load_dotenv()

for name in KEYS:
    print(name, "loaded:", bool(os.getenv(name)))


In [4]:
# %% [CELL 3] Goal, Task, Plan, NextAction, State
from typing import Any
from pydantic import BaseModel, Field
from langgraph.graph import MessagesState


class Goal(BaseModel):

    objective: str = Field(
        description="What the user ultimately wants to achieve."
    )

    requirements: list[str] = Field(
        description="Things that must be included or completed."
    )

    constraints: list[str] = Field(
        description="Limitations or conditions to follow."
    )

    context: str = Field(
        description="Additional information needed to understand the goal."
    )


class Task(BaseModel):

    task_id: str = Field(
        description="Unique ID of the task."
    )

    objective: str = Field(
        description="What this task needs to accomplish."
    )

    tool: str = Field(
        description="The tool required to execute this task."
    )

    depends_on: list[str] = Field(
        default_factory=list,
        description="IDs of tasks that must be completed first."
    )

    condition: str | None = Field(
        default=None,
        description="Optional condition for executing this task."
    )


class Plan(BaseModel):
    tasks: list[Task]


class NextAction(BaseModel):

    task_id: str = Field(
        description="ID of the next task to execute."
    )

    reason: str = Field(
        description="Reason for selecting this task."
    )


class State(MessagesState):

    query: str

    goal: Goal | None

    plan: Plan | None

    completed_tasks: list[str]

    executed_results: dict[str, Any]

    current_task: Task | None

    current_state: dict[str, Any]

    finished: bool

    final_answer: str


In [5]:
# %% [CELL 4] Gemini model
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

google_key = os.getenv("GOOGLE_API_KEY")

if not google_key:
    raise ValueError("GOOGLE_API_KEY is missing")

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=google_key,
    max_retries=6,
)


In [6]:
# # %% [CELL 5] Model name test (jawab aaye to model theek hai)
# print(model.invoke("Say hello in one word.").content)

In [7]:
# %% [CELL 6] Tools
import os
import ast
import json
import math
import operator
import requests

from dotenv import load_dotenv
from tavily import TavilyClient
from langchain_core.tools import tool

load_dotenv()



False

In [8]:
# =====================================
# TOOL 1: Dubai Weather
# =====================================

@tool
def get_dubai_weather() -> str:
    """Get the current weather in Dubai, including temperature,
    humidity, wind speed, and weather conditions."""

    api_key = os.getenv("OPENWEATHER_API_KEY")

    if not api_key:
        return "Error: OPENWEATHER_API_KEY is missing."

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "lat": 25.2048,
        "lon": 55.2708,
        "appid": api_key,
        "units": "metric",
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=15,
        )
        response.raise_for_status()

        data = response.json()

        result = {
            "city": "Dubai",
            "temperature_celsius": data["main"]["temp"],
            "feels_like_celsius": data["main"]["feels_like"],
            "humidity_percent": data["main"]["humidity"],
            "wind_speed_m_s": data["wind"]["speed"],
            "conditions": data["weather"][0]["description"],
        }

        return json.dumps(result, ensure_ascii=False)

    except (requests.RequestException, KeyError, IndexError) as e:
        return f"Weather tool failed: {type(e).__name__}"



In [9]:

# =====================================
# TOOL 2: Web Search
# =====================================

@tool
def search_web(query: str) -> str:
    """Search the web for current information, official ticket
    prices, attractions, and trustworthy source URLs."""

    api_key = os.getenv("TAVILY_API_KEY")

    if not api_key:
        return "Error: TAVILY_API_KEY is missing."

    try:
        client = TavilyClient(api_key=api_key)

        response = client.search(
            query=query,
            search_depth="basic",
            max_results=5,
        )

        results = []

        for item in response.get("results", [])[:5]:
            results.append({
                "title": item.get("title"),
                "url": item.get("url"),
                # FIX: 700 se 1500 (price cut na ho)
                "content": (item.get("content") or "")[:1500],
            })

        return json.dumps(results, ensure_ascii=False)

    except Exception as e:
        return f"Web search failed: {type(e).__name__}"



In [10]:

@tool
def get_aed_to_pkr_rate() -> str:
    """Get the latest published AED to PKR reference exchange
    rate and its publication date."""

    url = "https://api.frankfurter.dev/v2/rate/aed/pkr"

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        data = response.json()

        return json.dumps({
            "base_currency": data["base"],
            "target_currency": data["quote"],
            "rate": data["rate"],
            "date": data["date"],
            "source": "Frankfurter",
        }, ensure_ascii=False)

    except (requests.RequestException, KeyError, ValueError) as e:
        return f"Currency tool failed: {type(e).__name__}"



In [11]:
# =====================================
# TOOL 4: Safe Python Calculator
# =====================================

_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
}


def _safe_calculate(node):

    if isinstance(node, ast.Expression):
        return _safe_calculate(node.body)

    if (
        isinstance(node, ast.Constant)
        and type(node.value) in (int, float)
    ):
        return node.value

    if isinstance(node, ast.UnaryOp):
        if isinstance(node.op, ast.UAdd):
            return +_safe_calculate(node.operand)

        if isinstance(node.op, ast.USub):
            return -_safe_calculate(node.operand)

    if isinstance(node, ast.BinOp):
        operation = _ALLOWED_OPERATORS.get(type(node.op))

        if operation is None:
            raise ValueError("Only +, -, * and / are allowed.")

        left = _safe_calculate(node.left)
        right = _safe_calculate(node.right)

        if isinstance(node.op, ast.Div) and right == 0:
            raise ValueError("Division by zero is not allowed.")

        result = operation(left, right)

        if not math.isfinite(result) or abs(result) > 1e15:
            raise ValueError("Result is outside the allowed range.")

        return result

    raise ValueError("Invalid arithmetic expression.")


@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression using numbers and
    the +, -, * and / operators. Do not include currency symbols."""

    try:
        if len(expression) > 100:
            return "Calculation error: Expression is too long."

        tree = ast.parse(expression, mode="eval")
        result = _safe_calculate(tree)

        return json.dumps({
            "expression": expression,
            "result": round(result, 4),
        })

    except (ValueError, SyntaxError, TypeError, ZeroDivisionError) as e:
        return f"Calculation error: {e}"


In [12]:
# =====================================
# Register all tools
# =====================================

tools = [
    get_dubai_weather,
    search_web,
    get_aed_to_pkr_rate,
    calculator,
]


In [13]:
# # %% [CELL 7] Tools test (internet + keys check)
# r = requests.get("https://api.frankfurter.dev/v2/rate/aed/pkr", timeout=15)
# print(r.status_code, r.json())
# print(get_dubai_weather.invoke({}))


In [14]:
# %% [CELL 8] Goal node
from langchain_core.messages import SystemMessage, HumanMessage

goal_model = model.with_structured_output(
    Goal,
    method="function_calling",
)


def goal_node(state: State) -> dict:

    query = state["query"]

    messages = [
        SystemMessage(
            content="""
            You are a goal extraction assistant.

            Understand the user's request and extract:
            1. The main objective.
            2. The requirements that must be fulfilled.
            3. Any constraints explicitly mentioned by the user.
            4. Additional context available in the query.

            Do not execute any tools.
            Do not invent requirements or constraints.
            If no constraints are specified, return an empty list.
            """
        ),
        HumanMessage(
            content=f"User query:\n{query}"
        ),
    ]

    goal = goal_model.invoke(messages)

    return {
        "goal": goal
    }


In [15]:
# %% [CELL 9] Planner node (FIX: plan validation + 3 retries)
from langchain_core.messages import SystemMessage, HumanMessage

plan_model = model.with_structured_output(
    Plan,
    method="function_calling",
)


def planner_node(state: State) -> dict:

    query = state["query"]
    goal = state["goal"]

    if goal is None:
        raise ValueError("Goal is missing. Run goal_node first.")

    system_prompt = """
    You are a planning assistant for a sequential AI agent.

    Create a Plan containing tasks required to achieve the user's goal.

    Available tools:
    1. get_dubai_weather: Get current weather in Dubai (takes no arguments).
    2. search_web: Search the web for current information and ticket prices.
    3. get_aed_to_pkr_rate: Get the published AED to PKR exchange rate.
    4. calculator: Calculate mathematical expressions.

    Rules:
    - Every task must have a unique task_id.
    - The tool field must match one of the available tool names exactly.
    - Use depends_on to specify which tasks must complete first.
    - Weather, ticket search, and exchange-rate tasks are independent.
    - The AED ticket-total task depends on the ticket-price search.
    - The PKR conversion task depends on both the AED total
      and the exchange-rate task.
    - Do not invent ticket prices or exchange rates.
    - Use the calculator for arithmetic.
    - You MUST include calculator tasks in the plan: one for the AED total
      (ticket price x 2 adults) and one for the PKR total
      (AED total x exchange rate). Calculations must be part of the plan.
    - Only the final written answer is excluded from the plan.
    - Keep task objectives clear and actionable.
    - Do not create a task for writing the final answer.
      The final_answer_node handles the final recommendation.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(
            content=(
                f"User query:\n{query}\n\n"
                f"Extracted goal:\n{goal.model_dump_json(indent=2)}"
            )
        ),
    ]

    valid_tool_names = {t.name for t in tools}

    def check_plan(plan):
        """Plan theek ho to None return karta hai, warna error text."""

        if plan is None or not plan.tasks:
            return "Plan is empty."

        ids = [t.task_id for t in plan.tasks]

        if len(ids) != len(set(ids)):
            return "Duplicate task IDs."

        for t in plan.tasks:
            if t.tool not in valid_tool_names:
                return f"Unknown tool '{t.tool}' in task {t.task_id}."

            missing = set(t.depends_on) - set(ids)
            if missing:
                return f"Task {t.task_id} depends on missing tasks: {missing}."

        return None

    plan = None
    last_error = "unknown"

    for attempt in range(3):
        try:
            candidate = plan_model.invoke(messages)
            problem = check_plan(candidate)

            if problem is None:
                plan = candidate
                break

            last_error = problem

        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"

    if plan is None:
        raise ValueError(f"Planner failed after 3 attempts: {last_error}")

    return {
        "plan": plan
    }


Key '$defs' is not supported in schema, ignoring


In [16]:
# %% [CELL 10] Next action node (FIX: failed dependency skip + LLM call bachat)
from langchain_core.messages import SystemMessage, HumanMessage

next_action_model = model.with_structured_output(
    NextAction,
    method="function_calling",
)


def next_action_node(state: State) -> dict:

    plan = state["plan"]
    completed_tasks = set(state["completed_tasks"])
    executed_results = state.get("executed_results") or {}

    if plan is None:
        raise ValueError("Plan is missing. Run planner_node first.")

    # IDs of tasks whose tool call failed
    failed_ids = {
        task_id
        for task_id, result in executed_results.items()
        if isinstance(result, dict) and result.get("failed")
    }

    # Step 1: Find tasks that are not completed
    remaining_tasks = [
        task
        for task in plan.tasks
        if task.task_id not in completed_tasks
    ]

    # Step 2: Check whether all tasks are completed
    if not remaining_tasks:
        return {
            "finished": True,
            "current_task": None,
            "current_state": {
                **(state.get("current_state") or {}),
                "status": "finished",
                "reason": "All planned tasks are completed.",
            },
        }

    # Step 3: Ready = dependencies completed AND none of them failed
    ready_tasks = [
        task
        for task in remaining_tasks
        if set(task.depends_on).issubset(completed_tasks)
        and not (set(task.depends_on) & failed_ids)
    ]

    # Step 4: Detect a blocked plan
    if not ready_tasks:
        return {
            "finished": True,
            "current_task": None,
            "current_state": {
                **(state.get("current_state") or {}),
                "status": "blocked",
                "reason": "Remaining tasks depend on failed or unfinished tasks.",
            },
        }

    ready_tasks_by_id = {
        task.task_id: task
        for task in ready_tasks
    }

    # Step 5: Only one ready task -> no LLM call needed
    if len(ready_tasks) == 1:
        selected_task = ready_tasks[0]
        reason = "Only one task is ready."

    else:
        ready_tasks_json = [
            task.model_dump()
            for task in ready_tasks
        ]

        messages = [
            SystemMessage(
                content="""
                You are the Next Action selector for a sequential AI agent.

                Select exactly ONE task from the supplied ready tasks.

                Rules:
                - Select only a task whose task_id appears in ready_tasks.
                - Do not select completed tasks.
                - Do not invent a task_id.
                - Choose a task that helps achieve the user's goal.
                - Explain briefly why you selected it.
                """
            ),
            HumanMessage(
                content=f"""
                User query:
                {state["query"]}

                Current goal:
                {state["goal"].model_dump_json()}

                Completed task IDs:
                {list(completed_tasks)}

                Ready tasks:
                {ready_tasks_json}
                """
            ),
        ]

        try:
            action = next_action_model.invoke(messages)
            selected_task = ready_tasks_by_id.get(
                action.task_id, ready_tasks[0]
            )
            reason = action.reason
        except Exception:
            selected_task = ready_tasks[0]
            reason = "LLM selection failed; first ready task used."

    # Step 6: Update the State
    return {
        "current_task": selected_task,
        "finished": False,
        "current_state": {
            **(state.get("current_state") or {}),
            "status": "task_selected",
            "selected_task_id": selected_task.task_id,
            "reason": reason,
        },
    }


In [17]:
# %% [CELL 11] Tool agent node
import json
from langchain_core.messages import SystemMessage, HumanMessage

model_with_tools = model.bind_tools(tools)


def tool_agent_node(state: State) -> dict:

    # Step 1: Get the selected task
    task = state["current_task"]

    if task is None:
        raise ValueError("No current task is selected.")

    # Step 2: Check the assigned tool
    available_tool_names = {
        t.name for t in tools
    }

    if task.tool not in available_tool_names:
        raise ValueError(
            f"Unknown tool: {task.tool}"
        )

    # Step 3: Prepare previous results
    executed_results = state.get("executed_results", {})

    # Step 4: Give instructions to the LLM
    system_prompt = f"""
    You are the Tool Agent in a sequential AI workflow.

    Your job is to execute the currently selected task by
    generating a tool call.

    The planner has assigned this tool:
    {task.tool}

    Available tools:
    {list(available_tool_names)}

    Rules:
    - Call exactly the assigned tool.
    - Generate appropriate arguments for that tool.
    - Use previous task results when needed.
    - Do not invent missing prices, exchange rates, or data.
    - Do not answer the user directly.
    - Let the tool return the actual result.
    - For calculator: use only plain numbers (no commas, no currency symbols,
      no words) taken from previous task results, with +, -, *, /.
    - For search_web: write a specific English query,
      for example "Burj Khalifa At The Top ticket price AED".
    """

    human_prompt = f"""
    User query:
    {state["query"]}

    Current goal:
    {state["goal"].model_dump_json(indent=2)}

    Current task ID:
    {task.task_id}

    Current task objective:
    {task.objective}

    Assigned tool:
    {task.tool}

    Previous task results:
    {json.dumps(executed_results, indent=2, default=str)}
    """

    # Step 5: Ask the LLM to generate a tool call
    response = model_with_tools.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt),
    ])

    # Step 6: Store the AI response in graph state
    # (ghalat tool call ko route_after_tool_agent tool_error par bhej deta hai)
    return {
        "messages": [response]
    }



In [18]:
# %% [CELL 12] Router after tool agent
def route_after_tool_agent(state: State) -> str:

    last_message = state["messages"][-1]
    task = state["current_task"]

    tool_calls = getattr(last_message, "tool_calls", None)

    # Sirf tab "tools" jab assigned tool hi call hua ho
    if (
        tool_calls
        and len(tool_calls) == 1
        and tool_calls[0]["name"] == task.tool
    ):
        return "tools"

    return "tool_error"



In [19]:
# %% [CELL 13] Tool error node (FIX: retry, phir task failed mark)
MAX_TOOL_RETRIES = 2


def tool_error_node(state: State) -> dict:

    current_state = state.get("current_state") or {}
    task = state.get("current_task")

    task_id = task.task_id if task is not None else "unknown"

    retries = dict(current_state.get("tool_retries") or {})
    retries[task_id] = retries.get(task_id, 0) + 1

    update = {
        "current_state": {
            **current_state,
            "status": "tool_call_failed",
            "reason": "The LLM did not generate the correct tool call.",
            "tool_retries": retries,
        }
    }

    # Retries khatam: task ko failed mark karo taake workflow aage barhe
    if retries[task_id] > MAX_TOOL_RETRIES and task is not None:

        executed_results = dict(state.get("executed_results") or {})
        executed_results[task_id] = {
            "tool": task.tool,
            "result": "Error: tool call could not be generated.",
            "failed": True,
        }

        completed_tasks = list(state.get("completed_tasks") or [])
        if task_id not in completed_tasks:
            completed_tasks.append(task_id)

        update["executed_results"] = executed_results
        update["completed_tasks"] = completed_tasks
        update["current_state"]["status"] = "task_failed"

    return update



In [20]:
# %% [CELL 14] Tool node (sirf ek bar)
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)


# %% [CELL 15] Record result node
from langchain_core.messages import ToolMessage


def record_result_node(state: State) -> dict:

    # Step 1: Get the current task
    task = state["current_task"]

    if task is None:
        raise ValueError("No current task found.")

    # Step 2: Find the latest ToolMessage
    tool_message = next(
        (
            message
            for message in reversed(state["messages"])
            if isinstance(message, ToolMessage)
        ),
        None,
    )

    if tool_message is None:
        raise ValueError("No ToolMessage found.")

    # Step 3: Save the executed result
    executed_results = dict(
        state.get("executed_results") or {}
    )

    # Tool ka error result pakdo
    failed = str(tool_message.content).startswith((
        "Error:",
        "Weather tool failed",
        "Web search failed",
        "Currency tool failed",
        "Calculation error",
    ))

    executed_results[task.task_id] = {
        "tool": tool_message.name,
        "result": tool_message.content,
        "failed": failed,
    }

    # Step 4: Mark the task as completed
    completed_tasks = list(
        state.get("completed_tasks") or []
    )

    if task.task_id not in completed_tasks:
        completed_tasks.append(task.task_id)

    # Step 5: Update the current workflow status
    current_state = {
        **(state.get("current_state") or {}),
        "status": "task_failed" if failed else "task_completed",
        "last_completed_task": task.task_id,
        "last_result": tool_message.content,
    }

    # Step 6: Return updates to LangGraph State
    return {
        "executed_results": executed_results,
        "completed_tasks": completed_tasks,
        "current_state": current_state,
    }



In [21]:
# %% [CELL 16] Replanner node (FIX: sirf task fail hone par LLM call)
import json
from langchain_core.messages import SystemMessage, HumanMessage

replanner_model = model.with_structured_output(
    Plan,
    method="function_calling",
)


def _replanner_core(state: State) -> dict:

    # Step 1: Get the existing plan
    existing_plan = state["plan"]

    if existing_plan is None:
        raise ValueError("Plan is missing.")

    # Step 2: Get completed tasks and their results
    completed_tasks = set(state.get("completed_tasks") or [])
    executed_results = state.get("executed_results") or {}

    # Step 3: Get the current goal
    goal = state["goal"]

    # Step 4: Prepare instructions for the LLM
    system_prompt = """
    You are a Replanner for a sequential AI agent.

    Review the existing plan using the execution results.

    Your responsibilities:
    - Check whether the existing plan still achieves the goal.
    - Keep the existing plan if it is still valid.
    - Modify, add, remove, or replace future tasks when necessary.
    - Preserve every completed task and its original task_id.
    - Do not repeat completed tasks.
    - Keep task IDs unique.
    - Make sure every dependency refers to a task in the plan.
    - A task must depend on another task when it needs that task's result.
    - Use only the available tools.
    - Do not invent API results, ticket prices, or exchange rates.
    - Return the complete updated Plan, not an explanation.
    """

    available_tools = [t.name for t in tools]

    human_prompt = f"""
    User query:
    {state["query"]}

    Goal:
    {goal.model_dump_json(indent=2)}

    Existing plan:
    {existing_plan.model_dump_json(indent=2)}

    Completed task IDs:
    {json.dumps(list(completed_tasks))}

    Execution results:
    {json.dumps(executed_results, indent=2, default=str)}

    Current workflow state:
    {json.dumps(state.get("current_state") or {}, indent=2, default=str)}

    Available tools:
    {json.dumps(available_tools)}

    Review this information and return the complete updated Plan.
    """

    # Step 5: Ask the LLM to review the plan
    updated_plan = replanner_model.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt),
    ])

    # Step 6: Preserve completed tasks exactly as they were
    completed_task_objects = [
        task
        for task in existing_plan.tasks
        if task.task_id in completed_tasks
    ]

    # Keep only future tasks proposed by the replanner
    future_tasks = [
        task
        for task in updated_plan.tasks
        if task.task_id not in completed_tasks
    ]

    # Replanner ne baqi tasks hata diye to purane rakho
    remaining_existing = [
        task
        for task in existing_plan.tasks
        if task.task_id not in completed_tasks
    ]
    if remaining_existing and not future_tasks:
        future_tasks = remaining_existing

    # Step 7: Combine completed tasks and updated future tasks
    final_tasks = completed_task_objects + future_tasks

    # Step 8: Validate unique task IDs
    task_ids = [task.task_id for task in final_tasks]

    if len(task_ids) != len(set(task_ids)):
        raise ValueError("Replanner returned duplicate task IDs.")

    # Step 9: Validate dependencies
    valid_task_ids = set(task_ids)

    for task in final_tasks:
        invalid_dependencies = (
            set(task.depends_on) - valid_task_ids
        )

        if invalid_dependencies:
            raise ValueError(
                f"Task {task.task_id} has invalid dependencies: "
                f"{invalid_dependencies}"
            )

    # Future tasks mein sirf valid tool names allowed
    valid_tool_names = {t.name for t in tools}
    for task in final_tasks:
        if task.tool not in valid_tool_names:
            raise ValueError(f"Task {task.task_id} uses unknown tool: {task.tool}")

    # Step 10: Build the final Plan
    final_plan = Plan(tasks=final_tasks)

    # Step 11: Update graph state
    return {
        "plan": final_plan,
        "current_task": None,
        "current_state": {
            **(state.get("current_state") or {}),
            "status": "replanned",
            "completed_tasks_count": len(completed_tasks),
            "remaining_tasks_count": len(future_tasks),
        },
    }


def replanner_node(state: State) -> dict:

    status = (state.get("current_state") or {}).get("status")

    # Task theek chala to plan valid hai, LLM call ki zarurat nahi
    if status != "task_failed":
        return {"current_task": None}

    try:
        return _replanner_core(state)
    except Exception as e:
        return {
            "current_task": None,
            "current_state": {
                **(state.get("current_state") or {}),
                "status": "replan_failed",
                "reason": f"{type(e).__name__}: {e}",
            },
        }



Key '$defs' is not supported in schema, ignoring


In [22]:
# %% [CELL 17] Final answer node (FIX: typo "pthe heat")
import json
import re
from typing import Literal

from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage


# ============================================================
# 1. STRUCTURED FACTS MODEL
# ============================================================

class FinalFacts(BaseModel):

    ticket_name: str | None = Field(
        default=None,
        description="Ticket category or package name."
    )

    price_aed_per_adult: float | None = Field(
        default=None,
        description=(
            "Clearly stated price in AED for one adult. "
            "Return null if no reliable price is available."
        )
    )

    source_url: str | None = Field(
        default=None,
        description=(
            "Exact URL from the supplied search results "
            "that supports the selected ticket price."
        )
    )

    price_note: str | None = Field(
        default=None,
        description=(
            "Important price note, such as starting price, "
            "time-slot variation, or package details."
        )
    )

    decision: Literal[
        "Yes, go ahead",
        "Yes, with conditions",
        "Not now"
    ]

    reasons: list[str] = Field(
        description="Two or three short reasons supporting the decision."
    )

    tips: list[str] = Field(
        description="One or two practical visiting tips."
    )


facts_model = model.with_structured_output(
    FinalFacts,
    method="function_calling"
)


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def _get_ok(executed: dict, tool_name: str):
    """Find the saved result for a particular successful tool."""

    for item in executed.values():

        if not isinstance(item, dict):
            continue

        if (
            item.get("tool") == tool_name
            and not item.get("failed")
        ):
            return item.get("result")

    return None


def _parse_json(value):
    """Convert a JSON string into a Python object."""

    if isinstance(value, (dict, list)):
        return value

    if not isinstance(value, str) or not value.strip():
        return None

    try:
        return json.loads(value)

    except (json.JSONDecodeError, TypeError, ValueError):
        return None


def _fmt(value: float) -> str:
    """Format numbers with thousands separators and two decimals."""

    return f"{value:,.2f}"


def _clean_url(value):
    """Convert a plain URL or Markdown-style URL into a plain URL."""

    if not isinstance(value, str):
        return None

    value = value.strip()

    # Handles strings such as [https://example.com](https://example.com)
    match = re.search(r"\]\((https?://[^)]+)\)", value)

    if match:
        value = match.group(1)

    elif value.startswith("<") and value.endswith(">"):
        value = value[1:-1].strip()

    if not value.startswith(("https://", "http://")):
        return None

    return value.rstrip("., ")


# ============================================================
# 3. MAIN REPORT BUILDER
# ============================================================

def _final_answer_core(state: State) -> dict:

    executed = state.get("executed_results") or {}
    missing = []

    # --------------------------------------------------------
    # A. Extract weather data
    # --------------------------------------------------------

    weather = _parse_json(
        _get_ok(executed, "get_dubai_weather")
    )

    if not isinstance(weather, dict):
        weather = None
        missing.append("Weather data was not available.")

    # --------------------------------------------------------
    # B. Extract currency exchange rate
    # --------------------------------------------------------

    fx = _parse_json(
        _get_ok(executed, "get_aed_to_pkr_rate")
    )

    if not isinstance(fx, dict):
        fx = None

    rate = None

    if fx is not None:

        try:
            candidate_rate = float(fx["rate"])

            if candidate_rate > 0:
                rate = candidate_rate

        except (KeyError, TypeError, ValueError):
            rate = None

    if rate is None:
        missing.append(
            "AED to PKR exchange rate was not available."
        )

    # --------------------------------------------------------
    # C. Collect web search results
    # --------------------------------------------------------

    search_items = []

    for item in executed.values():

        if not isinstance(item, dict):
            continue

        if (
            item.get("tool") == "search_web"
            and not item.get("failed")
        ):

            parsed_items = _parse_json(
                item.get("result")
            )

            # The search tool normally returns a list.
            if isinstance(parsed_items, list):
                search_items.extend(
                    x for x in parsed_items
                    if isinstance(x, dict)
                )

            # Also support a response dictionary containing results.
            elif (
                isinstance(parsed_items, dict)
                and isinstance(parsed_items.get("results"), list)
            ):
                search_items.extend(
                    x for x in parsed_items["results"]
                    if isinstance(x, dict)
                )

    if not search_items:
        missing.append(
            "No usable web search results were found for ticket prices."
        )

    # URLs actually returned by the search tool
    valid_sources = {}

    for item in search_items:

        clean_url = _clean_url(item.get("url"))

        if clean_url:
            valid_sources[clean_url] = item

    search_text = "\n\n".join(
        (
            f"[{i}] {item.get('title', 'Untitled')}\n"
            f"URL: {_clean_url(item.get('url')) or 'Unavailable'}\n"
            f"Content: {str(item.get('content') or '')[:1500]}"
        )
        for i, item in enumerate(search_items, start=1)
    )

    if not search_text:
        search_text = "No search results available."

    weather_text = (
        json.dumps(weather, ensure_ascii=False)
        if weather is not None
        else "Weather data unavailable."
    )

    rate_text = (
        f"1 AED = {rate:.6f} PKR"
        if rate is not None
        else "Exchange rate unavailable."
    )

    # --------------------------------------------------------
    # D. LLM extracts facts and makes a recommendation
    # --------------------------------------------------------

    system_prompt = """
    You are a careful professional travel analyst.

    Extract facts and produce a recommendation using ONLY the
    supplied user query, weather data, exchange rate and search results.

    Ticket price rules:
    - Extract a price only when it is clearly stated in AED.
    - Never invent or guess a price.
    - Prefer the official Emaar listing for standard At The Top admission.
    - Do not confuse standard admission with combo packages, rooftop
      dining, Sky Views or other bundled experiences.
    - If only a starting price is available, clearly say it is a starting price.
    - If a reliable adult price cannot be identified, return null.
    - source_url must be copied from the supplied search results.

    Recommendation rules:
    - Consider the actual temperature, feels-like temperature,
      humidity and weather conditions.
    - Consider ticket-price certainty and the available budget information.
    - Choose only one of the three allowed decisions.
    - If important information is missing, use a cautious decision.
    - Do not claim that a price or weather measurement is verified
      unless it is supported by the supplied tool data.
    - If feels-like temperature is 35°C or higher,
      or humidity is 75% or higher, do not describe the weather
      as comfortable. Prefer 'Yes, with conditions' and explain
      the heat and humidity precautions.

    Writing rules:
    - Reasons and tips must be short, professional English.
    - Do not calculate totals or currency conversions.
    - Python will perform all arithmetic.
    """

    human_prompt = f"""
    USER QUERY:
    {state.get("query", "")}

    CURRENT WEATHER DATA:
    {weather_text}

    AED TO PKR EXCHANGE RATE:
    {rate_text}

    WEB SEARCH RESULTS:
    {search_text}
    """

    try:

        facts = facts_model.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=human_prompt),
        ])

        if not isinstance(facts, FinalFacts):
            raise TypeError(
                "The LLM did not return a valid FinalFacts object."
            )

    except Exception as e:

        print(
            "Final facts extraction failed:",
            type(e).__name__,
            str(e)
        )

        facts = FinalFacts(
            decision="Yes, with conditions",
            reasons=[
                "Some information could not be processed reliably."
            ],
            tips=[
                "Confirm the ticket category, date and final price "
                "on the official website before booking."
            ],
        )

    # --------------------------------------------------------
    # E. Validate ticket price and source URL
    # --------------------------------------------------------

    price = facts.price_aed_per_adult

    if price is not None and not (0 < price < 5000):
        price = None

    selected_url = _clean_url(facts.source_url)

    # Accept the LLM's URL only if it came from a search result.
    url = (
        selected_url
        if selected_url in valid_sources
        else None
    )

    if price is None:
        missing.append(
            "A clear, usable adult ticket price was not identified."
        )

    elif url is None:
        missing.append(
            "The selected ticket-price source could not be matched "
            "to a URL in the search results."
        )

    # --------------------------------------------------------
    # F. Calculate totals in Python
    # --------------------------------------------------------

    total_aed = None
    total_pkr = None

    if price is not None:

        # Two adult tickets.
        total_aed = price * 2

        # Convert total AED into PKR only if the rate is available.
        if rate is not None:
            total_pkr = total_aed * rate

    # --------------------------------------------------------
    # G. Build the Markdown report
    # --------------------------------------------------------

    lines = []

    lines.append("# Burj Khalifa Visit — Trip Report\n")

    lines.append(
        "> Weather, ticket pricing and budget assessment "
        "for two adults.\n"
    )

    decision_icons = {
        "Yes, go ahead": "✅",
        "Yes, with conditions": "⚠️",
        "Not now": "❌",
    }

    icon = decision_icons[facts.decision]

    lines.append(
        f"## {icon} Recommendation: {facts.decision}\n"
    )

    for reason in facts.reasons[:3]:
        lines.append(f"- {reason}")

    lines.append("")

    # Weather
    lines.append("## 🌤️ Current Weather in Dubai\n")

    if weather:

        lines.append("| Metric | Value |")
        lines.append("|---|---:|")

        temp = weather.get("temperature_celsius")

        if temp is not None:
            lines.append(
                f"| Temperature | {float(temp):.1f} °C |"
            )

        feels_like = weather.get("feels_like_celsius")

        if feels_like is not None:
            lines.append(
                f"| Feels like | {float(feels_like):.1f} °C |"
            )

        humidity = weather.get("humidity_percent")

        if humidity is not None:
            lines.append(
                f"| Humidity | {humidity}% |"
            )

        wind = weather.get("wind_speed_m_s")

        if wind is not None:
            wind_kmh = float(wind) * 3.6

            lines.append(
                f"| Wind speed | {wind_kmh:.1f} km/h |"
            )

        conditions = weather.get(
            "conditions",
            "Unavailable"
        )

        lines.append(
            f"| Conditions | {str(conditions).capitalize()} |"
        )

    else:

        lines.append(
            "Weather information is currently unavailable."
        )

    lines.append("")

    # Ticket price
    lines.append("## 🎟️ Ticket Price\n")

    if price is not None:

        ticket_name = (
            facts.ticket_name
            or "At The Top, Burj Khalifa"
        )

        lines.append(
            f"- **{ticket_name}:** "
            f"{_fmt(price)} AED per adult"
        )

        if facts.price_note:
            lines.append(
                f"- **Price note:** {facts.price_note}"
            )

        if url:
            lines.append(
                f"- **Price source:** [{url}]({url})"
            )

        lines.append(
            "- Confirm the exact ticket category, date and time slot "
            "before booking."
        )

    else:

        lines.append(
            "A reliable adult ticket price was not confirmed. "
            "Please check the official ticket website before booking."
        )

    lines.append("")

    # Budget
    lines.append("## 💰 Budget for 2 Adults\n")

    if total_aed is not None:

        lines.append("| Calculation | Amount |")
        lines.append("|---|---:|")

        lines.append(
            f"| Price per adult | {_fmt(price)} AED |"
        )

        lines.append(
            f"| Two tickets ({_fmt(price)} × 2) | "
            f"**{_fmt(total_aed)} AED** |"
        )

        if rate is not None and total_pkr is not None:

            lines.append(
                f"| Exchange rate | 1 AED = {rate:.4f} PKR |"
            )

            lines.append(
                f"| Total in PKR ({_fmt(total_aed)} × {rate:.4f}) | "
                f"**{_fmt(total_pkr)} PKR** |"
            )

        else:

            lines.append(
                "| Total in PKR | Cannot calculate: exchange rate unavailable |"
            )

    else:

        lines.append(
            "The ticket total cannot be calculated because "
            "a reliable adult ticket price is unavailable."
        )

    lines.append("")

    lines.append(
        "*This estimate covers ticket admission only. It excludes "
        "transport, food and other travel expenses.*"
    )

    lines.append("")

    # Practical tips
    if facts.tips:

        lines.append("## 💡 Practical Tips\n")

        for tip in facts.tips[:2]:
            lines.append(f"- {tip}")

        lines.append("")

    # Sources
    lines.append("## 📚 Sources\n")

    if weather:

        lines.append(
            "- **Weather:** [OpenWeather Current Weather API]"
            "(https://openweathermap.org/api/current)"
        )

    else:

        lines.append("- **Weather:** Not available")

    if url:

        lines.append(
            f"- **Ticket price:** [{url}]({url})"
        )

    elif search_items:

        lines.append(
            "- **Ticket research:** The search returned results, "
            "but a specific price source could not be validated."
        )

        for item in search_items[:5]:

            source_url = _clean_url(item.get("url"))

            if source_url:

                title = str(
                    item.get("title") or "Ticket search result"
                )

                lines.append(
                    f"  - [{title}]({source_url})"
                )

    if rate is not None and fx:

        currency_source = fx.get(
            "source",
            "Currency API"
        )

        currency_url = _clean_url(
            fx.get("source_url")
        )

        if not currency_url:

            if "frankfurter" in str(currency_source).lower():

                currency_url = (
                    "https://api.frankfurter.dev/v2/rate/aed/pkr"
                )

            else:

                currency_url = (
                    "https://www.exchangerate-api.com/docs/free"
                )

        update_date = (
            fx.get("last_updated_utc")
            or fx.get("date")
            or "Date not provided"
        )

        lines.append(
            f"- **Exchange rate:** [{currency_source}]"
            f"({currency_url}) — updated: {update_date}"
        )

    else:

        lines.append(
            "- **Exchange rate:** Not available"
        )

    lines.append("")

    # Missing information
    if missing:

        lines.append("## ⚠️ Missing Information\n")

        for item in dict.fromkeys(missing):
            lines.append(f"- {item}")

        lines.append("")

    # Disclaimer
    lines.append("---")

    lines.append(
        "*Weather is live data, while exchange rates are published "
        "reference rates. Ticket prices can vary by category and "
        "time slot. Confirm the final ticket price before booking.*"
    )

    return {
        "final_answer": "\n".join(lines)
    }


# ============================================================
# 4. SAFETY WRAPPER
# ============================================================

def final_answer_node(state: State) -> dict:

    try:

        output = _final_answer_core(state)

        final_answer = output.get(
            "final_answer",
            ""
        )

        if isinstance(final_answer, str) and final_answer.strip():
            return output

        reason = "The report builder returned an empty answer."

    except Exception as e:

        reason = f"{type(e).__name__}: {e}"

        print(
            "Final answer generation failed:",
            reason
        )

    return {
        "final_answer": (
            "## ⚠️ Final Report Error\n\n"
            f"The final report could not be generated: `{reason}`\n\n"
            "Please inspect the saved tool results and error message."
        )
    }



In [23]:
# %% [CELL 18] Build graph (FIX: tool_error -> next_action retry)
from langgraph.graph import StateGraph, START, END

# Create graph
builder = StateGraph(State)


# Register all nodes
builder.add_node("goal", goal_node)
builder.add_node("planner", planner_node)
builder.add_node("next_action", next_action_node)
builder.add_node("tool_agent", tool_agent_node)
builder.add_node("tools", tool_node)
builder.add_node("record_result", record_result_node)
builder.add_node("replanner", replanner_node)
builder.add_node("tool_error", tool_error_node)
builder.add_node("final", final_answer_node)


# Initial fixed edges
builder.add_edge(START, "goal")
builder.add_edge("goal", "planner")
builder.add_edge("planner", "next_action")


# Route after selecting the next task
def route_after_next_action(state: State) -> str:

    if state["finished"]:
        return "finish"

    return "execute"


builder.add_conditional_edges(
    "next_action",
    route_after_next_action,
    {
        "execute": "tool_agent",
        "finish": "final",
    },
)


# Route after the LLM generates a tool call
builder.add_conditional_edges(
    "tool_agent",
    route_after_tool_agent,
    {
        "tools": "tools",
        "tool_error": "tool_error",
    },
)


# After tool execution, record its result
builder.add_edge("tools", "record_result")


# Replan check after recording the result
builder.add_edge("record_result", "replanner")


# Select the next task from the (updated) plan
builder.add_edge("replanner", "next_action")


# FIX: ghalat tool call par retry (pehle seedha final par jata tha)
builder.add_edge("tool_error", "next_action")


# End after generating the final answer
builder.add_edge("final", END)


# Compile the graph
graph = builder.compile()


# %% [CELL 19] Run the agent
from langchain_core.messages import HumanMessage
from IPython.display import Markdown, display

query = """
Check the current weather in Dubai.
Search for the latest ticket prices for Burj Khalifa At The Top
and find the current AED to PKR exchange rate.
Then calculate the total cost of tickets for 2 adults in both AED and PKR.
Considering the weather conditions, ticket prices and total budget,
give a recommendation on whether the visit is suitable.
Mention the sources for every result and explain the calculations.
"""

initial_state = {
    "messages": [HumanMessage(content=query)],
    "query": query,
    "goal": None,
    "plan": None,
    "completed_tasks": [],
    "executed_results": {},
    "current_task": None,
    "current_state": {"status": "started"},
    "finished": False,
    "final_answer": "",
}

result = graph.invoke(
    initial_state,
    config={"recursion_limit": 100},
)

display(Markdown(result["final_answer"]))

# Burj Khalifa Visit — Trip Report

> Weather, ticket pricing and budget assessment for two adults.

## ⚠️ Recommendation: Yes, with conditions

- Feels-like temperature reaches 37.84°C, requiring heat and sun precautions despite indoor observation decks.
- Standard adult ticket prices start at AED 159.0 per adult, which is clearly identified from source material.
- Total costs can be calculated reliably for 2 adults in both AED and PKR.

## 🌤️ Current Weather in Dubai

| Metric | Value |
|---|---:|
| Temperature | 33.0 °C |
| Feels like | 37.8 °C |
| Humidity | 55% |
| Wind speed | 9.3 km/h |
| Conditions | Overcast clouds |

## 🎟️ Ticket Price

- **At The Top (Levels 124 & 125):** 159.00 AED per adult
- **Price note:** Starting price for At The Top (Levels 124 & 125) general admission, varying by time slot.
- **Price source:** [https://venusdestination.com/burj-al-khalifa-at-the-top-tickets-how-to-get-to-the-top-of-burj-khalifa](https://venusdestination.com/burj-al-khalifa-at-the-top-tickets-how-to-get-to-the-top-of-burj-khalifa)
- Confirm the exact ticket category, date and time slot before booking.

## 💰 Budget for 2 Adults

| Calculation | Amount |
|---|---:|
| Price per adult | 159.00 AED |
| Two tickets (159.00 × 2) | **318.00 AED** |
| Exchange rate | 1 AED = 75.5530 PKR |
| Total in PKR (318.00 × 75.5530) | **24,025.85 PKR** |

*This estimate covers ticket admission only. It excludes transport, food and other travel expenses.*

## 💡 Practical Tips

- Book tickets in advance to secure preferred time slots and avoid peak-hour surcharges.
- Stay hydrated and take breaks in air-conditioned areas while moving between Dubai Mall and the observation deck.

## 📚 Sources

- **Weather:** [OpenWeather Current Weather API](https://openweathermap.org/api/current)
- **Ticket price:** [https://venusdestination.com/burj-al-khalifa-at-the-top-tickets-how-to-get-to-the-top-of-burj-khalifa](https://venusdestination.com/burj-al-khalifa-at-the-top-tickets-how-to-get-to-the-top-of-burj-khalifa)
- **Exchange rate:** [Frankfurter](https://api.frankfurter.dev/v2/rate/aed/pkr) — updated: 2026-10-13

---
*Weather is live data, while exchange rates are published reference rates. Ticket prices can vary by category and time slot. Confirm the final ticket price before booking.*